# Humanizer: teach an editor when to stop

A LoRA adapter on GLM 5.3 Flash that rewrites AI-sounding prose so it reads naturally, keeps every fact, and leaves
already-good prose alone.

**Headline:** on 99 held-out passages the adapter keeps every fact literal in 100% of its rewrites [95% interval 96 to
100%], where the same base model given the full Humanizer skill text keeps them in 94% [87 to 97%]; see the first
figure in [section 1](#1.-See-the-result-first). On pass rate the adapter ties the prompted base, so its claim is
preservation and restraint, not a higher score.

**Needs:** Python 3.12; from `cookbook/training` run
`uv venv --python 3.12 .venv-humanizer && uv pip install --python .venv-humanizer/bin/python jupyterlab ipykernel matplotlib`
(the full launch steps are in `GETTING_STARTED.md`); open Jupyter from this `sft_humanizer` folder so `walkthrough/`,
`pack/` and `results/` are beside the notebook; `../common/figures.py` (the gallery's shared plotting code) is imported.

**Status:** Recorded results are read from committed files; Run All makes no API call and spends nothing. The paid
knobs, both off by default and both needing `FIREWORKS_API_KEY` in the environment of the Jupyter process:

- `RUN_TRAINING` (section 8): one serverless SFT job. The smoke preset costs cents; the tutorial preset is on the
  order of a dollar.
- `RUN_EVALUATION` (section 9): paid generation and model judging for base and your adapter. Five development
  passages: cost not measured, small; the full 99-passage test run with the second pass: a few dollars. Needs a
  promoted adapter loaded onto serverless with `firectl` (see section 9).

**Three routes:** Explore the result (sections 1 to 5) · Reproduce it (sections 6 to 9, paid) · Use your own data
(section 10).

Recorded observations are from October 2026, seed 29, and are not a forecast of your run. From section 7 on, cards
about your own run are marked in orange; recorded evidence stays in grey.

**Glossary**

- **Adapter / LoRA rank:** a LoRA adapter is a small set of extra weights trained on top of a frozen base model; the
  rank (here 64) is the width of those weights, and alpha (here 128) is a fixed scale applied to them.
- **SFT:** supervised fine-tuning, training on example inputs and the answers we want.
- **Base vs prompted vs adapter arm:** an arm is one way of producing a rewrite. Base is GLM 5.3 Flash with a
  two-sentence editor instruction; prompted (the teacher) is the same model given the full Humanizer skill text; the
  adapter is base weights plus the LoRA, with only the editor instruction.
- **Teacher / student prompt:** the teacher is base plus the full skill text, which wrote the training targets; the
  student prompt is the short editor instruction the adapter is trained and called with.
- **Held-out:** the 99 test passages the adapter never saw in training.
- **Greedy draw:** one generation at temperature 0, so each arm gives one answer per passage at temperature zero.
- **Preserve / pass rate:** preserve is a mechanical check that every number, name and protected span survives; pass
  adds a model judge scoring at least 0.8 on meaning, additions, naturalness and following the request.
- **Over-edit:** a sequence-similarity score below 0.85 on an already-good passage, measured on the 11 held-out passages of class good.
- **Second pass (idempotence):** running an arm on its own output; a real editor should find nothing left to do, so a
  sequence-similarity score below 0.90 counts as a change.
- **Wilson interval:** a 95% interval for a yes/no rate that stays sensible at 0% and 100%; drawn as whiskers on bars.
- **Paired delta / bootstrap:** the difference between two arms on the same passages, with a 95% interval from
  resampling those passages together, which is the only figure here that tests a difference.
- **Checkpoint vs promoted model:** training saves a checkpoint; promotion registers it as a model in your account that
  can be called by name.
- **Serverless vs dedicated deployment:** serverless is the shared public endpoint, billed per token; a dedicated
  deployment is GPUs reserved for you. The recorded seed runs used a dedicated deployment that served several
  adapters at once (multi-LoRA); the 2026-10-12 re-run and the reader route use serverless.
- **reasoning_effort:** a request setting that caps how much the model thinks before answering; the editor is called
  with `low`.

## 1. See the result first

Before reading how the adapter was built, look at what it does to one passage. Below is a held-out source, the rewrite
from the base model driven by the full Humanizer skill text (the teacher), and the rewrite from the adapter given only a
two-sentence editor instruction. Highlighted spans are the words that differ from the source.


In [ ]:
from pathlib import Path
import json, sys, subprocess

HERE = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "walkthrough/recorded.json").exists()), None)
if HERE is None:
    raise RuntimeError("Open the notebook from the sft_humanizer folder with walkthrough/recorded.json beside it.")
sys.path.insert(0, str(HERE))
from walkthrough.cards import ARM_LABELS, card, compare_card, recorded_note
from walkthrough.humanizer_figures import (plt, style, arm_bars, delta_bars, two_panel, save, wilson_ci,
                                           paired_delta_ci, headline_figure, lighten_plain, rate_ci,
                                           UP, DOWN, METRICS, LEFT, RIGHT)

style()
FIG_DIR = HERE / "figures"
# The recorded lesson: 99 held-out passages from the seed-29 run, every arm's output and per-row scores. No API calls.
lesson = json.loads((HERE / "walkthrough/recorded.json").read_text())
cases = {row["id"]: row for row in lesson["cases"]}
RECORDED = recorded_note(lesson)
compare_card(cases, "hz-0142", note=RECORDED)

The adapter made the smaller edit and kept every number, and the rest of the lesson is doing this on all 99 passages.
The Humanizer story is four numbers per arm: how often every fact literal survives (preserve), how often an
already-good passage gets rewritten anyway (over-edit), how often a second pass over the arm's own output changes it
again (a real editor should find nothing left to do), and the pass rate, which adds a model judge at 0.8 on meaning,
additions, naturalness and on-request.

The figure reads the committed `summary.json` from the serverless re-run of 2026-10-12, where the seed-29 adapter and
the two prompted arms were called through the public endpoint. Every metric is a per-passage yes/no, so each bar gets a
95% Wilson interval from its count and denominator; the denominators are on the axis labels, and the over-edit rate is
measured on only the 11 passages of class `good`, which is why its interval is wide. The per-row `samples.jsonl` for
this run is committed beside the summary, so the next cell pairs the adapter against the full-skill prompt passage by
passage; section 3 repeats the test on the recorded run.
`headline_figure` lives in `walkthrough/humanizer_figures.py`, beside the rest of this notebook's plotting code.

In [ ]:
serverless = json.loads((HERE / "results/2026-10-12/serverless-eval-s29/summary.json").read_text())["summary"]
fig = headline_figure(serverless, FIG_DIR / "humanizer_result.png",
                      suptitle="Does the adapter keep the facts and stop editing where the prompts do not?")
plt.show()

In [ ]:
# Paired test on the serverless run: same 99 passages, adapter minus the full-skill prompt, per-row from samples.jsonl.
rows = [json.loads(l) for l in (HERE / "results/2026-10-12/serverless-eval-s29/samples.jsonl").read_text().splitlines() if l.strip()]
by_arm = {}
for r in rows:
    by_arm.setdefault(r["arm"], {})[r["id"]] = r
for metric in ("pass", "preserve"):
    pairs = [(float(by_arm["skill"][i][metric]), float(by_arm["adapter_s29"][i][metric])) for i in by_arm["adapter_s29"] if i in by_arm["skill"]]
    m, lo, hi = paired_delta_ci(pairs)
    print(f"{metric:8s} adapter - full skill: {100*m:+.1f} points [{100*lo:+.1f}, {100*hi:+.1f}]  (n={len(pairs)} passages)")


**What to see:** the paired numbers above are the serverless answer to "is the adapter better than the prompt?". A
lower bound above zero means the adapter beat the full skill prompt on the same passages; an interval that includes zero
means this run cannot tell them apart on that metric.

In [ ]:
# The reader's tell: did the rewrite add an em dash the author never wrote? Counted per row from samples.jsonl.
from build_humanizer_pack import em_dash_added
inputs = {r["id"]: r for r in (json.loads(l) for l in (HERE / "data/humanizer_inputs.jsonl").read_text().splitlines() if l.strip())}
print(f"{'arm':30s} {'added em dash':>14s} {'removed one':>12s}   (of {len(by_arm['adapter_s29'])} passages; {sum(1 for i in by_arm['adapter_s29'] if em_dash_added(inputs[i]['source'], ''))} sources contain one)")
for arm in ("base_plain", "base_short", "skill", "adapter_s29"):
    rs = by_arm[arm]
    added = sum(em_dash_added(r["text"], inputs[r["id"]]["source"]) for r in rs.values())
    removed = sum(1 for r in rs.values() if em_dash_added(inputs[r["id"]]["source"], "") and not em_dash_added(r["text"], ""))
    print(f"{ARM_LABELS.get(arm, arm):30s} {added:14d} {removed:12d}")


**What to see:** the base model puts an em dash into most rewrites whose source had none; the adapter adds none and
takes most of the author's own out, the same as the full skill prompt. Nothing in the training data mentioned dashes:
the teacher answers simply did not contain them, and the filter now rejects a teacher answer that adds one
(`em_dash_added` in `build_humanizer_pack.py`; the shipped pack already passes). The evaluator reports the rate as
`em dash added` beside preserve and pass.

**What to see:** the adapter's preserve bar reaches 100% and its second-pass bar sits near 0%, while its pass bar is level with the editor-instruction base.

Yes on preservation and restraint, no on pass rate: serverless, seed-29 adapter and prompts on the same 99 held-out
passages; preserve is the mechanical check that every fact literal survives, pass adds the judge at 0.8 on all four
axes, over-edit is a change of more than 15% to an already-good passage, and 2nd pass is a change of more than 10% when
the arm edits its own output. The adapter's 100% preservation still carries an interval of about 96 to 100%, its pass
rate (91%, 84 to 95) is not separable from the editor-instruction base (92%, 85 to 96), and the over-edit interval on
11 good passages (9%, 2 to 38) still allows the adapter and the teacher (36%, 15 to 65) to be closer than the bars
suggest; the paired test is in section 3.

Three seeds were trained to check that the recipe, not the seed, produced this. Seeds 17 and 43 were evaluated against
base with the short instruction in the same run as each adapter, on a dedicated deployment serving several adapters at once; base moves a point
or two between runs because the judge is not fully deterministic.

In [ ]:
SEEDS = {
    17: "results/2026-10-09/pack-eval-s17/summary.json",
    29: "results/2026-10-08/pack-eval-s29/summary.json",
    43: "results/2026-10-09/pack-eval-s43/summary.json",
}
runs = {seed: json.loads((HERE / path).read_text())["summary"] for seed, path in SEEDS.items()}

fig, axes = plt.subplots(2, 2, figsize=(11, 7.6))
for ax, (key, label, n) in zip(axes.flat, METRICS):
    series = []
    for arm_label, pick in [("Base, editor instruction", lambda s, seed: s["base_short"]),
                            ("Adapter, editor instruction", lambda s, seed: s[f"adapter_s{seed}"])]:
        vals, los, his = zip(*(rate_ci(pick(runs[seed], seed)[key], n) for seed in SEEDS))
        series.append((arm_label, vals, los, his))
    direction = "higher is better" if (key, label, n) in UP else "lower is better"
    arm_bars(ax, [f"seed {seed}" for seed in SEEDS], series, title=label.replace("\n", " "),
             ylabel=f"% of {n} passages ({direction})", fmt="{:.0f}%", ylim=(0, 118), legend=False)
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.01))
fig.suptitle("Is it the recipe or the seed?")
fig.tight_layout(rect=(0, 0.04, 1, 1))
save(fig, FIG_DIR / "humanizer_three_seeds.png")
plt.show()


**What to see:** on every seed the adapter's second-pass bar sits below the base's whiskers; its preservation and
over-edit bars point the same way as seed 29's, but their whiskers overlap the base's.

The recipe: three adapters (seeds 17, 29, 43) against base with the editor instruction, 99 held-out passages each on a
dedicated deployment, 95% Wilson intervals from the counts. Second-pass change is 0 to 2% on every seed, with intervals
(at most 0 to 7%) clear of base's 16 to 22% (10 to 31%). Preservation point estimates are 97 to 99% against base's
89 to 90%, but only seed 29's interval (94 to 100%) clears base's (81 to 94%); seeds 17 and 43 (92 to 99%) overlap it.
Over-edit on the 11 good passages is 9 to 18% against 73%, and those intervals (2 to 48% against 43 to 90%) are too
wide to call a separation on their own. Pass rate is 86 to 94% against 86 to 89%. These are per-arm intervals, not a
test of the difference; the paired comparison on the one run with per-row data follows in section 3. The GUIDE.md has
the full tables, the judge scores and the per-class breakdown.

## 2. Read another rewrite, and one that was left alone

Aggregate rates hide what an editor actually did to a sentence, so the rest of the recorded sections work at the level of
single passages. Change `CASE_ID` to read any of the 99. `hz-0430` is a passage the adapter left unchanged, which is the
correct output for already-good prose; `hz-0233` is the adapter's instructive failure, used in the next section.


In [ ]:
CASE_ID = "hz-0430"  # Try hz-0233 (meaning-check failure) or any id from `cases`.
compare_card(cases, CASE_ID, note=RECORDED, arms=("base_short", "skill", "adapter_s29"))


## 3. Predict, then inspect the checks

Judging a rewrite yourself before looking at the scores is the best calibration you can get for free. A pass requires no
mechanical problems (every fact literal present, no new numbers or URLs, sane length, no preamble) and every judge score
at least 0.8. The judge scores meaning, additions, naturalness, and whether the output follows the request.

Read `hz-0233` below and decide which arms pass before looking at the chart. The adapter's rewrite drops the explicit
statement that the fix was **not a new framework**: its mechanical checks pass, but its meaning score is 0.70. That
interpretation is an editorial reading of the two texts; the snapshot contains scores, not a judge rationale.


In [ ]:
CASE_ID = "hz-0233"
case = cases[CASE_ID]
compare_card(cases, CASE_ID, note=RECORDED, arms=("base_short", "adapter_s29"))

AXES = ["meaning_preserved", "no_additions", "natural", "on_request"]
scored, unscored = [], []
for arm, result in case["arms"].items():
    if result["judge"] is None:
        unscored.append(arm)
    else:
        scored.append((ARM_LABELS[arm], [result["judge"][a] for a in AXES]))
checks = ", ".join(f"{ARM_LABELS[a]}: " + ("no problems" if not r["problems"] else "; ".join(r["problems"]))
                   for a, r in case["arms"].items())

fig, ax = plt.subplots(figsize=(10, 4.2))
arm_bars(ax, [a.replace("_", " ") for a in AXES], scored,
         title=f"Which arms clear the judge on passage {CASE_ID}?",
         ylabel="judge score (0 to 1), pass threshold 0.8", fmt="{:.2f}", ylim=(0, 1.15), baseline=0.8)
lighten_plain(fig, [label for label, _ in scored])
fig.tight_layout()
plt.show()
print("Mechanical checks.", checks)
if unscored:
    print("Unscored judge result (not a failure):", ", ".join(ARM_LABELS[a] for a in unscored))
print("These are historical model judgments, not human gold labels.")


**What to see:** only the editor-instruction base's bars all clear the 0.8 line; the adapter's meaning bar stops at 0.70.

Only base with the editor instruction clears 0.8 on every axis: the adapter (0.70) and the teacher (0.75) both miss on
meaning while passing every mechanical check, which is exactly the failure those checks cannot see (no literal was lost,
a negation was), and the plain prompt misses on additions instead (recorded seed-29 run, judge scores per axis). One
passage is an anecdote; the figure below recomputes the four metrics for all four recorded arms from the per-row scores
with Wilson intervals. It includes the plain-prompt base arm that the headline leaves out; one of that arm's 99 rows
(`hz-0122`) has an unscored judge result, which is excluded from its pass rate and not counted as a failure, so that
arm's pass rate is over 98 passages.

In [ ]:
def per_row(arm, key):
    rows = [c for c in lesson["cases"] if key != "over_edit" or c["cls"] == "good"]
    if key == "pass":
        rows = [c for c in rows if c["arms"][arm]["judge"] is not None]
    if key == "idem_changed":
        return [int((c["arms"][arm]["idem_similarity"] or 0) < 0.9) for c in rows]
    return [int(bool(c["arms"][arm][key])) for c in rows]

RECORDED_KEYS = {"preserve_rate": "preserve", "over_edit_good": "over_edit",
                 "idem_changed": "idem_changed", "pass_rate": "pass"}
def recorded_series(metrics):
    series = []
    for arm in lesson["arms"]:
        stats = []
        for key, _, _ in metrics:
            rows = per_row(arm, RECORDED_KEYS[key])
            stats.append(wilson_ci(sum(rows), len(rows)))
        series.append((ARM_LABELS[arm], [100 * p for p, _, _ in stats],
                       [100 * lo for _, lo, _ in stats], [100 * hi for _, _, hi in stats]))
    return series

fig = two_panel(groups=[label for _, label, _ in UP], series=recorded_series(UP), left=LEFT, right=RIGHT,
                right_series=recorded_series(DOWN), figsize=(13, 4.8),
                suptitle="Does the pattern hold on the recorded run, with a plain-prompt base added?")
fig.axes[1].set_xticks(range(len(DOWN)), [label for _, label, _ in DOWN])
lighten_plain(fig, [label for label, *_ in recorded_series(UP)])
save(fig, FIG_DIR / "humanizer_recorded_arms.png")
plt.show()


**What to see:** the same ordering as the serverless figure, with the light-grey plain-prompt base worst on every bar.

It holds, and the plain prompt shows what "no editor instruction" costs (recorded seed-29 run of 2026-10-08, four arms
on the same 99 passages, 95% Wilson intervals): base with a bare "rewrite this" rewrites 91% of good prose and changes a
third of its own output on a second pass, while the adapter is at 99% preservation and 1% second-pass change. Per-arm
intervals do not test the difference between arms; because this run has per-row scores for every arm on the same
passages, the next figure asks that question directly with a paired bootstrap of adapter minus prompt (the glossary defines it).

In [ ]:
COMPARISONS = [("skill", "full skill"), ("base_short", "editor instruction")]
labels, deltas, los, his = [], [], [], []
for ref, short in COMPARISONS:
    for key, name in [("pass", "Pass"), ("preserve", "Preserve")]:
        pairs = [(int(c["arms"][ref][key]), int(c["arms"]["adapter_s29"][key])) for c in lesson["cases"]]
        m, lo, hi = paired_delta_ci(pairs)
        labels.append(f"{name}\nvs {short}")
        deltas.append(100 * m); los.append(100 * lo); his.append(100 * hi)

fig, ax = plt.subplots(figsize=(9, 4.4))
delta_bars(ax, labels, deltas, los, his, title="Is the adapter better than the prompts on the same passages?",
           ylabel="adapter minus prompt, points\n(95% paired bootstrap)")
ax.set_ylim(min(los) - 4, max(his) + 9)
fig.tight_layout()
save(fig, FIG_DIR / "humanizer_paired_delta.png")
plt.show()
for label, m, lo, hi in zip(labels, deltas, los, his):
    print(f"{label.replace(chr(10), ' ')}: {m:+.1f} points [{lo:+.1f}, {hi:+.1f}]")


**What to see:** three of the four whiskers stay above zero; the fourth (pass rate against the editor instruction) crosses it.

Against the full skill text the adapter is better on both counts: pass rate +16 points (95% paired interval +7 to +25)
and preservation +8 points (+3 to +14), the same 99 passages resampled together. Against base with the editor
instruction, preservation is +10 points (+4 to +17) and pass rate is +1 point (-8 to +10), an interval that includes
zero and a 8-point loss, so "same editor on pass rate" is the only claim the data supports.

## 4. Check your understanding

The failure mode this adapter was trained against is subtle: an edit that reads better but says something stronger or
narrower than the source. This small authored exercise, not an output of the recorded experiment, checks that you can spot it.

Source: "The pilot **may** reduce latency by **about 20%**, but we have **not tested it under peak load**."

Choose one rewrite by setting `choice` below:

- **a:** "The pilot reduces latency by 20%, including under peak load."
- **b:** "The pilot may cut latency by about 20%. We haven't tested it under peak load."
- **c:** "The pilot may cut latency by about 20%."


In [ ]:
choice = ""  # a, b, or c; predict before revealing the explanation.
feedback = {
    "a": "No: uncertainty became certainty, and a peak-load result was invented.",
    "b": "Yes: the uncertainty, the approximate size and the untested condition all survive.",
    "c": "No: the peak-load limitation is lost. A shorter edit can still change the meaning.",
}
print(feedback.get(choice, "Choose a, b, or c above, then rerun this cell."))


## 5. Understand one training row

The adapter learns from examples, so the shape of one example explains most of the recipe. The teacher (base model plus
the full skill text) produces candidate rewrites; mechanical and judge filters choose a target. Supervised fine-tuning (SFT) then learns from
**student instruction + input, then accepted answer**. Only the assistant answer is the target, and the full skill text is
never inserted into the student prompt. An accepted target is not guaranteed perfect, so read this one critically too.

In [ ]:
example = lesson["training_example"]
messages = example["messages"]
user_text = next(m["content"] for m in messages if m["role"] == "user")
source_text = user_text.split("Text:\n", 1)[-1]
blocks = []
for m in messages:
    if m["role"] == "assistant":
        blocks.append(("Assistant (the training target, edits highlighted against the source)", m["content"], source_text))
    else:
        blocks.append((m["role"].capitalize() + (" (the student prompt)" if m["role"] == "system" else " (the input)"),
                       m["content"]))
card("One training row from pack/humanizer_train.jsonl", blocks,
     note=f"Recorded: one of {lesson['pack_counts']['train']} rows; each JSONL line is {{\"messages\": [...]}} with these three turns")


## 6. Validate the frozen pack locally

A training run is only as reproducible as its data, so the lesson pins the pack by hash before any plan is made. The pack
has 203 training rows, 49 development inputs and 99 test inputs. Use the development set while changing the recipe and
evaluate the untouched test set only after selecting the recipe. This structural validation cannot decide whether a
target is a faithful edit.


In [ ]:
import hashlib
from train_walkthrough import validate_dataset, make_plan

pack = HERE / "pack/humanizer_train.jsonl"
rows = validate_dataset(pack)
assert hashlib.sha256(pack.read_bytes()).hexdigest() == lesson["provenance"]["train_sha256"], "The pack changed since this lesson snapshot. Review the change before proceeding."
print(f"{len(rows)} valid training rows; hash matches the recorded pack.")
print("Roles, nonempty text, and duplicate inputs checked. Meaning still requires review.")


## 7. Prepare a live run, still free

From here on the notebook is about **your run**, not the recorded one; nothing above is evidence that your run worked.
Seeing the exact plan before paying for it removes most surprises. The smoke preset uses eight examples for one epoch
(about two optimizer steps) and checks training mechanics, not quality. The tutorial preset uses the full pack for two
epochs (about 102 steps) with the current shared SFT recipe; it is **not an exact reproduction** of the historical
100-step run. Hyperparameters are LoRA rank 64, alpha 128, learning rate 1e-4, batch size 4 (see the glossary for rank and alpha).

`OUTPUT_MODEL_ID` below names the model that a finished run is promoted to. Leave it empty for the smoke; for a tutorial
run you want to compare in section 9 it must be set **before** training, because an unnamed run saves a checkpoint
that cannot be evaluated.

The command below prints a plan without importing paid SDK clients or writing run files.

In [ ]:
PRESET = "smoke"  # Choose "tutorial" after completing the mechanics smoke.
RUN_DIR = HERE / "results" / f"lesson-{PRESET}"
OUTPUT_MODEL_ID = ""  # NEW model ID to promote to. Empty = smoke only: the adapter cannot be evaluated in section 9.
command = [sys.executable, str(HERE / "train_walkthrough.py"),
           "--preset", PRESET, "--run-dir", str(RUN_DIR)]
if OUTPUT_MODEL_ID:
    command += ["--output-model-id", OUTPUT_MODEL_ID]
card("Your run: local plan only, no API calls", [("Preset", PRESET), ("Run directory", str(RUN_DIR))],
     note="Not recorded evidence. The plan below is printed by train_walkthrough.py for this checkout.", kind="live")
subprocess.run(command, check=True)

## 8. Optional: train with your account

This is the first paid step, so it is behind its own switch. First install the cookbook's dependencies in the same
Python 3.12 environment as this kernel. The commands assume your shell is in `cookbook/training` (two levels up from
this folder), where the `.venv-humanizer` environment and the package's `pyproject.toml` live:

```bash
cd cookbook/training
uv pip install --python .venv-humanizer/bin/python -e .
.venv-humanizer/bin/python -m ipykernel install --user --name humanizer-lesson
# Select the humanizer-lesson kernel in Jupyter and rerun the preparation cells.
```

Supply `FIREWORKS_API_KEY` through your environment; do not paste it into a notebook cell. Confirm your account has
access to the selected training model and check current [training prices](https://fireworks.ai/pricing) before opting
in. The historical 100-step run was reported at about 15 minutes; this path has not been timed or billed in a fresh
external account.

**The knob:** `RUN_TRAINING = False` (the default) starts nothing and bills nothing; `True` submits one serverless SFT
job for the plan above. Cost: the smoke preset is cents, the tutorial preset on the order of a dollar (training bills
per training token; `GETTING_STARTED.md` has the current notes). Read `status.json` and `metrics.jsonl` in the run directory
during execution. Completion produces `lesson-result.json`; promotion, if requested, also writes model information.
Promotion does not verify serving availability. Everything the run writes lands under `results/`, which is gitignored,
so your evidence stays untracked unless you copy it elsewhere.

The wrapper refuses a nonempty run directory to prevent accidental replay. After an interruption, keep the directory and
checkpoint reference and consult the shared SFT recipe's checkpoint/resume instructions. Do not delete evidence just to
get past the guard.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    subprocess.run(command + ["--execute"], check=True)
else:
    card("Your run: training is off", [("Status", "RUN_TRAINING = False. No job was started and nothing was billed.")],
         note="Set RUN_TRAINING = True to start a paid serverless SFT job for the plan above.", kind="live")


## 9. Optional: compare the served adapter

A completed training run is not evidence of a better editor; only the comparison is. For a tutorial run you want to
evaluate, set `OUTPUT_MODEL_ID` to a new model resource name `accounts/YOUR_ACCOUNT/models/YOUR_NEW_MODEL` **before
training**. After success, `RUN_DIR / "output-model.json"` records `model_id`, `checkpoint` and `job_id`. A smoke with
no model ID saves a checkpoint but does not promote an inference model. See `GETTING_STARTED.md` for the
serving-readiness check and evaluation command.

Once your promoted adapter has a valid inference model name for your account, compare it with base plus the same short
instruction. Prerequisite: the promoted model must be loaded onto serverless first
(`firectl -a YOUR_ACCOUNT model load-lora YOUR_NEW_MODEL --serverless --wait`, about five minutes, `firectl` 1.8.15 or
newer); until then the endpoint answers `404`. `GETTING_STARTED.md` walks through the load, a one-request probe, and
how to free the slot afterwards. Do not assume an old preview deployment or someone else's account ID will work.

**The knob:** `RUN_EVALUATION = False` (the default) calls no model; `True` runs the evaluator. Start with five
**development** passages. The evaluator runs a model-judge calibration gate, then generates and judges both arms; these
are paid inference calls separate from training. Cost: five passages is small and not measured; the full 99-passage test
run with the second pass is a few dollars. The small smoke skips the second edit. When ready,
set `EVAL_SPLIT = "test"`, `MAX_INPUTS = 0` and `CHECK_IDEMPOTENCE = True` for the final held-out comparison. Each
execution reserves a new evaluation directory and prints its path; earlier evidence is never overwritten.

Expected files: `samples.jsonl` (inspect the text) and `summary.json` / `summary.md` (aggregate rates). The next
cell draws your own `summary.json` in the style of section 1, in its own figure: the recorded bars above are not your
result. Note that `results/` is gitignored, so this summary is untracked. A high mechanical rate is not proof of semantic preservation; compare meaning
scores, failures and changes to already-good prose before making a quality claim.

In [ ]:
from evaluate_walkthrough import run_evaluation

RUN_EVALUATION = False
ADAPTER_MODEL = ""  # Use your own serving-ready inference name; see GETTING_STARTED.md.
EVAL_SPLIT = "dev"
MAX_INPUTS = 5  # 0 means all passages in the selected split.
CHECK_IDEMPOTENCE = False
if RUN_EVALUATION:
    eval_dir = run_evaluation(
        ADAPTER_MODEL, RUN_DIR, split=EVAL_SPLIT,
        max_inputs=MAX_INPUTS, idempotence=CHECK_IDEMPOTENCE,
    )
    print("Inspect your results in:", eval_dir)
else:
    card("Your run: evaluation is off", [("Status", "RUN_EVALUATION = False. No model was called.")],
         note="The recorded comparisons in sections 1 to 3 are historical and say nothing about your adapter.", kind="live")


In [ ]:
# Draw your own evaluation in the section 1 style. Uncomment and point MY_SUMMARY at the summary.json the cell above
# printed (for example RUN_DIR / "eval-test-<stamp>-<suffix>" / "summary.json"). Reads a local file; no API call.
# MY_SUMMARY = RUN_DIR / "eval-test-REPLACE-ME" / "summary.json"
# mine = json.loads(MY_SUMMARY.read_text())["summary"]
# fig = headline_figure(mine, RUN_DIR / "my_result.png", arms=["base_short", "adapter"],
#                       suptitle="Your run: base with the editor instruction vs your adapter (not the recorded result)")
# plt.show()

## 10. Adapt it to your writing

The recipe transfers to your own prose only through your own examples. Create a separate JSONL file with one `messages`
row per input, using the system/user/assistant shape from section 5. Start with a handful of manually reviewed pairs,
keep the audience and editor instruction at inference time, and keep test passages out of training.

Set a path below to check the file locally. Validation checks structure, not semantic correctness. After validation,
pass `--dataset your-file.jsonl` to `train_walkthrough.py` and inspect its plan.


In [ ]:
MY_DATASET = ""  # Optional path to your chat JSONL. This cell never trains.
if MY_DATASET:
    own_rows = validate_dataset(Path(MY_DATASET).expanduser())
    print(f"Structure passed: {len(own_rows)} rows. Review meaning and reserve evaluation inputs next.")
else:
    print("Add a JSONL path when you are ready to bring your own examples.")


## What to take away

**Did training help?** Yes, on the two things the skill is for. The adapter keeps every fact literal (100% serverless,
97 to 99% across seeds), leaves good prose alone far more often than the prompted teacher (9 to 18% vs 36% over-edit)
and has nothing left to do on a second pass (0 to 2% vs 9 to 22%). Of these, the second-pass intervals are clear of
every prompted arm on every seed; preservation clears the editor-instruction base only for seed 29 and in the paired
test, and the over-edit intervals on 11 good passages are too wide to separate the arms on their own. Against the full skill text on the same passages it is +16 points on pass rate (+7 to +25
paired) and +8 on preservation (+3 to +14). It did not raise the pass rate over base with the editor instruction (+1
point, -8 to +10), so the honest claim is "same editor on pass rate, better on preservation and restraint".

**Where did it fail?** On meaning that no literal carries. hz-0233 keeps every number and drops a negation, and the
judge's meaning score for the adapter (0.90) sits below the editor-instruction base (0.95). The judge is the same model
family that filtered the pack, so "more natural than its teacher" is partly self-agreement; a human-graded subset is
still pending.

**Adapter, prompt, or more data?** If you only need the style on short paragraphs, the editor instruction alone is a
strong baseline (92% pass) and costs nothing to train. Take the adapter when preservation and restraint matter: long
passages with hedges, numbers and conditions, or a pipeline that re-edits its own output (the second-pass metric). Take more data before either
if your genre is not one of the five here; the pack does not show transfer to other genres, languages or long documents.

**One experiment to try next.** Add 20 to 40 reviewed pairs from your own genre in the section 5 shape, train the
tutorial preset, and evaluate on the dev split with `CHECK_IDEMPOTENCE = True`. The number to watch is over-edit on your
own already-good passages, not pass rate: if the adapter starts rewriting them, the targets taught it to edit, not to stop.

The companion page and this notebook read the same `walkthrough/recorded.json` snapshot. See its provenance hashes and
the detailed guide (`GUIDE.md`) for the full tables and limits.